# Classification Trees and Ensembles — Solution

**Dataset:** Titanic passengers: survival from age, fare, passenger class and sex.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, roc_auc_score

## 1. Calculate Gini and a leaf probability

Replace the gap with `1 - p ** 2 - (1 - p) ** 2`. Is a 40/60 node pure? Compare the two supplied splits: which reduces Gini most? For the separate 145/125 leaf, which class is predicted?

In [ ]:
p = 0.4
gini = 1 - p ** 2 - (1 - p) ** 2
print('Parent Gini:', gini)
# Split male?: 60 rows (12 survived), 40 rows (28 survived).
g_male_left = 1 - (12 / 60) ** 2 - (48 / 60) ** 2
g_male_right = 1 - (28 / 40) ** 2 - (12 / 40) ** 2
weighted_male = (60 / 100) * g_male_left + (40 / 100) * g_male_right
# Split first class?: 25 rows (15 survived), 75 rows (25 survived).
g_class_left = 1 - (15 / 25) ** 2 - (10 / 25) ** 2
g_class_right = 1 - (25 / 75) ** 2 - (50 / 75) ** 2
weighted_class = (25 / 100) * g_class_left + (75 / 100) * g_class_right
print('Gini decrease male?:', round(gini - weighted_male, 3))
print('Gini decrease first class?:', round(gini - weighted_class, 3))
leaf_probability = 145 / (145 + 125)
print('Leaf P(survived):', round(leaf_probability, 3))
print('Leaf class:', int(leaf_probability >= 0.5))

**Answer:** Gini is 0.48, so the parent is mixed. The male split decreases Gini by 0.12 versus about 0.027 for first class; choose male as the root split. The separate leaf has survivor probability about 0.537 and predicts survived.

## 2. Prepare Titanic data

Data preparation is provided. What does `male` mean? Why is `stratify=y` used? Note that excluding passengers with missing age may affect representativeness.

In [ ]:
titanic = pd.read_csv('../00_Data/titanic.csv')
# Fixed, simple preparation: use rows with known age and fare.
# Removing missing rows may change which passengers are represented.
titanic = titanic.dropna(subset=['Age', 'Fare']).copy()
X = titanic[['Age', 'Fare', 'Pclass']].copy()
X['male'] = (titanic['Sex'] == 'male').astype(int)
y = titanic['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.head())
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

**Answer:** male is 1 for men and 0 for women. Stratification keeps the survival class proportions similar in training and test sets.

## 3. Train and follow a small tree

Replace the gap with `DecisionTreeClassifier(max_depth=2, random_state=42)`. Follow the rules for the two passengers supplied below. Read the class names when interpreting each leaf.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)
plt.figure(figsize=(11, 5))
plot_tree(tree, feature_names=list(X.columns),
          class_names=['Not survived', 'Survived'], filled=True, rounded=True)
plt.show()
new_passengers = pd.DataFrame({'Age': [30, 30], 'Fare': [10, 80],
                               'Pclass': [3, 1], 'male': [1, 0]})
print('Predicted classes:', tree.predict(new_passengers))
print('P(survived):', tree.predict_proba(new_passengers)[:, 1].round(3))

**Answer:** Follow the thresholds to each final node. `predict_proba` reports the training survivor share in that leaf. A class label summarises that probability and does not mean certainty.

## 4. See the effect of pruning

The demonstration is supplied. Which setting creates fewer leaves? Why is the higher training accuracy not enough to select the tree?

In [ ]:
for alpha in [0.0, 0.01]:
    pruned = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    pruned.fit(X_train, y_train)
    print('alpha:', alpha, 'leaves:', pruned.get_n_leaves(),
          'training accuracy:', round(pruned.score(X_train, y_train), 3))

**Answer:** Positive alpha can remove branches and reduce training accuracy. An unpruned tree can memorise training rows. Use validation to select complexity.

## 5. Compare forest and boosting with CV

The models and CV are provided. Explain which ensemble averages independently built trees and which adds corrections sequentially. Compare CV AUC.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42)
boosting = GradientBoostingClassifier(n_estimators=100, max_depth=2,
                                     learning_rate=0.1, random_state=42)
for name, model in [('Forest', forest), ('Boosting', boosting)]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    print(name, 'CV AUC:', round(scores.mean(), 3))

**Answer:** The forest averages varied bootstrap trees; boosting adds small corrections in sequence. Larger CV AUC indicates better ranking on these validation folds. It does not establish that the method is best for all data.

## 6. Run the supplied search

Run this complete example. We use `max_features="sqrt"` versus `1.0` (all inputs). With our four inputs, a setting of 0.5 would also consider two inputs, exactly like sqrt; using 1.0 makes the comparison distinct. What are the chosen settings? How many setting combinations were tried? Why does the final confusion matrix use a threshold?

In [ ]:
grid = GridSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    {'max_features': ['sqrt', 1.0], 'min_samples_leaf': [1, 5, 20]},
    cv=5, scoring='roc_auc')
grid.fit(X_train, y_train)
print('Chosen settings:', grid.best_params_)
print('Best CV AUC:', round(grid.best_score_, 3))
p = grid.predict_proba(X_test)[:, 1]
print('Final test AUC:', round(roc_auc_score(y_test, p), 3))
print('Final confusion matrix:')
print(confusion_matrix(y_test, p >= 0.5, labels=[0, 1]))

**Answer:** There are 2 × 3 = 6 combinations, each assessed in five folds. The printed best settings come from training CV. A threshold converts survival probabilities into class labels for the confusion matrix.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))